## 1. Setup do ambiente

### 1.1 Importação de Bibliotecas
Importação de funções essenciais do **PySpark**, 
Selecionamos especificamente a função (`current_timestamp`) para a criação da coluna `data_criacao_silver` na tabela criada `chamados_geral`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp

catalogo = 'medalhao_credit'
bronze_db_name = 'bronze_credit'
silver_db_name = 'silver_credit' 

### 1.2 Configuração de Ambiente

Definição do catálogo (`medalhao_credit`) e o schema (`silver_credit`) que serão utilizados. 

In [0]:
%sql
USE CATALOG medalhao_credit;
USE SCHEMA silver_credit;

## 2. Tratamento da tabela `chamados_hora` 

### 2.1 Tabela `chamados_hora` na camada bronze

A tabela armazena informações sobre chamados de atendimento na camada bronze do Data Lake. A tabela contém os seguintes campos:

- **ID_Chamado**: Identificador único do chamado.
- **ID_Cliente**: Identificador do cliente relacionado ao chamado.
- **Hora_Abertura_Chamado**: Data e hora em que o chamado foi aberto.
- **Hora_Inicio_Atendimento**: Data e hora de início do atendimento do chamado.
- **Hora_Finalizacao_Atendimento**: Data e hora de finalização do atendimento.
- **data_ingestao**: Data de ingestão do registro na camada bronze.

In [0]:
df_ft_chamados_hora = spark.table(f'{catalogo}.{bronze_db_name}.chamados_hora')
df_ft_chamados_hora.limit(5).display()

### 2.2 Tratamento de nomes das colunas na tabela `chamados_hora`

- Os nomes das colunas do DataFrame foram convertidos para letras minúsculas, garantindo padronização.

In [0]:
df_ft_chamados_hora = df_ft_chamados_hora.select([F.col(c).alias(c.lower()) for c in df_ft_chamados_hora.columns])

### 2.3 Verificação / limpeza de dados da tabela `chamados_hora` 

- **Contagem inicial de linhas**: Verifica o número total de registros presentes no DataFrame `chamados_hora`.
- **Verificação de valores nulos**: Verifica linhas onde qualquer uma das colunas essenciais (`hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`, `data_ingestao`) possui valor nulo.
- **Verificação de linhas com tempos inconsistentes**: Verifica registros onde os cálculos de tempo (`tempo_espera_seg`, `tempo_atendimento_seg`, `diff_abertura_ingestao_seg`) resultam em valores negativos, indicando inconsistência temporal.
- **Verificação de valores irregulares em identificadores**: Verifica linhas onde os campos `id_cliente` e `id_chamado` são nulos ou não seguem o padrão numérico esperado.
- **Contagem final de linhas**: Exibe o total de registros restantes após todas as etapas de filtragem.

In [0]:
print(f'linhas em chamados_hora: {df_ft_chamados_hora.count()}')

In [0]:
df_ft_chamados_hora_null = df_ft_chamados_hora.filter(
    F.col('hora_abertura_chamado').isNull() |
    F.col('hora_inicio_atendimento').isNull() |
    F.col('hora_finalizacao_atendimento').isNull() |
    F.col('data_ingestao').isNull()
)

if df_ft_chamados_hora_null.count() == 0:
    print('valores nulos: 0')
else:
    print(f'linhas com val nulos:') 
    df_ft_chamados_hora_null.limit(5).display()

#### 2.3.1 Tratamento de valores na tabela `chamados_hora`

- As colunas `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento` passaram por duas etapas:
  1. Remoção de caracteres indesejados (" às ") usando `regexp_replace` para limpar os valores.
  2. Conversão dos valores dessas colunas para o tipo `timestamp`, utilizando o formato `'dd/MM/yyyy HH:mm:ss'`.

In [0]:
hora_cols = ['hora_abertura_chamado', 'hora_inicio_atendimento', 'hora_finalizacao_atendimento']

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.regexp_replace(F.col(col), r' �s ', ' ')
    )

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.to_timestamp(col, 'dd/MM/yyyy HH:mm:ss')
    )

#### 2.3.2 Criação de colunas na tabela `chamados_hora`

- **tempo_espera_seg**: tempo entre a abertura do chamado e o início do atendimento.
- **tempo_atendimento_seg**: tempo entre o início e a finalização do atendimento.
- **diff_abertura_ingestao_seg**: tempo entre a abertura do chamado e o momento de ingestão do registro na base.

Os cálculos são feitos convertendo os timestamps para o tipo `long` (segundos desde a época Unix) e subtraindo os valores correspondentes.

In [0]:
df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
    'tempo_espera_seg',
    (F.col('hora_inicio_atendimento').cast('long') - F.col('hora_abertura_chamado').cast('long'))
).withColumn(
    'tempo_atendimento_seg',
    (F.col('hora_finalizacao_atendimento').cast('long') - F.col('hora_inicio_atendimento').cast('long'))
).withColumn(
    'diff_abertura_ingestao_seg',
    (F.col('data_ingestao').cast('long') - F.col('hora_abertura_chamado').cast('long'))
)

In [0]:
# Verificação de linhas onde o tempo é inconsistente

df_ft_chamados_hora_tempo_dif = df_ft_chamados_hora.filter(
    (F.col('tempo_espera_seg') < 0) |
    (F.col('tempo_atendimento_seg') < 0) |
    (F.col('diff_abertura_ingestao_seg') < 0)
)

if df_ft_chamados_hora_tempo_dif.count() == 0:
    print('valores com tempo inconsistente: 0')
else:
    print(f'linhas com tempo inconsistente:')
    df_ft_chamados_hora_tempo_dif.limit(5).display()

In [0]:
# Verificação de valores irregulares em id_cliente e id_chamado

df_ft_chamados_hora_irreg = df_ft_chamados_hora.filter(
    F.col('id_cliente').isNull() &
    F.col('id_chamado').isNull() &
    ~F.col('id_cliente').rlike(r'^[0-9]+$') &
    ~F.col('id_chamado').rlike(r'^[0-9]+$')
)

if df_ft_chamados_hora_irreg.count() == 0:
    print('linhas com valores irregulares: 0')
else:
    print(f'linhas com id_cliente, id_chamado irregulares:')
    df_ft_chamados_hora_irreg.limit(5).display()

### 2.4 Remoção de registros inconsistentes

- Removemos da tabela (`df_ft_chamados_hora`) todos os registros com valores temporais inconsistentes.
- Essa etapa garante que apenas os chamados com dados temporais consistentes permaneçam para análise, eliminando registros com diferenças de tempo consideradas inválidas.

In [0]:
# df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_null)

print(f'linhas em chamados_hora apos remocao: {df_ft_chamados_hora.count()}')

### 2.5 Salvando dados tratados da tabela `chamados_hora` na camada silver

- O DataFrame `df_ft_chamados_hora`, após todas as etapas de limpeza e transformação, é salvo na tabela `chamados_hora` na camada silver utilizando o método `saveAsTable` com o modo `overwrite`, garantindo que os dados estejam atualizados.

In [0]:
df_ft_chamados_hora.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_hora')

df = spark.table(f'{catalogo}.{silver_db_name}.chamados_hora')
df.limit(5).display()


## Documentação da tabela chamados_geral

In [0]:
# leitura das tabelas

nomes_tabelas = [
  "base_atendentes", # id_atendente
  "base_motivos", # nome_motivo
  "canais", # nome_canal
  "chamados", # tabela usada como base para os joins
  "chamados_hora", # id_chamado
  "clientes", # id_cliente
  "custos", # id_chamado
  "pesquisa_satisfacao" # id_chamado
]

df_base_atendentes = spark.table(f"{catalogo}.{silver_db_name}.base_atendentes")
df_base_motivos = spark.table(f"{catalogo}.{silver_db_name}.base_motivos")
df_canais = spark.table(f"{catalogo}.{silver_db_name}.canais")
df_chamados = spark.table(f"{catalogo}.{silver_db_name}.chamados")
df_chamados_hora = spark.table(f"{catalogo}.{silver_db_name}.chamados_hora")
df_clientes = spark.table(f"{catalogo}.{silver_db_name}.clientes")
df_custos = spark.table(f"{catalogo}.{silver_db_name}.custos")
df_pesquisa_satisfacao = spark.table(f"{catalogo}.{silver_db_name}.pesquisa_satisfacao")

In [0]:
df_base_motivos.limit(5).display()

In [0]:
# criacao da tabela chamados_geral

df_chamados_geral = (
    df_chamados
    .join(df_chamados_hora, "id_chamado", "right")
    .join(df_base_atendentes, "id_atendente", "left")
    .join(df_base_motivos, df_chamados["motivo"] == df_base_motivos["nome_motivo"], "left")
    .join(df_canais, df_chamados["canal"] == df_canais["nome_canal"], "left")
    .join(df_clientes, "id_cliente", "left")
    .join(df_custos, "id_chamado", "left")
    .join(df_pesquisa_satisfacao, "id_chamado", "left")
    .select(
        "id_chamado",
        df_chamados["id_cliente"].alias("id_cliente"),
        "motivo",
        "categoria", # tudo nulo
        "categoria_nota",
        "nota_atendimento",
        "criticidade", # tudo nulo
        "canal",
        "status_canal",
        "resolvido",
        df_chamados_hora["hora_abertura_chamado"],
        df_chamados_hora["hora_inicio_atendimento"],
        df_chamados_hora["hora_finalizacao_atendimento"],
        "tempo_espera_segundos",
        "tempo_atendimento_segundos",
        "id_atendente",
        "nome_atendente",
        "nivel_atendimento",
        "valor_custo",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        current_timestamp().alias("data_criacao_silver")
    ).orderBy('id_chamado')
)

display(df_chamados_geral)

In [0]:
df_chamados_geral.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_geral')